# 02 · 检索编排：召回 → 重排

对应任务 **T8**，里程碑 **M2**。

这一节把 `vectorstore`（召回）与 `rerank`（精排）通过 `RAGPipeline` 串起来，
并亲眼观察**重排到底把哪些内容捞了上来**。

> 前置：`data/uploads/` 下至少有一篇 `.txt` / `.md`（P0-5）。
> 本 Notebook 会重建索引，重复运行不会累积重复 chunk（幂等）。

In [ ]:
import numpy as np

from kbqa.chunking import split_text
from kbqa.config import settings
from kbqa.embeddings import get_embedding_provider
from kbqa.llm import get_llm
from kbqa.loaders import list_documents, load_document
from kbqa.models import Chunk
from kbqa.pipeline import RAGPipeline
from kbqa.rerank import get_reranker
from kbqa.vectorstore import VectorStore

INDEX_DIR = settings.data_dir / "index"
print(f"召回 top_k = {settings.retrieve_top_k} ｜ 重排 top_n = {settings.rerank_top_n} ｜ 重排启用 = {settings.rerank_enabled}")

## 1. 准备索引

把 `data/uploads/` 下的每篇文档走一遍：切分 → 向量化 → 入库。

先 `remove_document` 再 `add`，保证**幂等**——重复运行不会累积重复 chunk。

In [ ]:
docs = list_documents(settings.data_dir)
print(f"uploads 下共 {len(docs)} 篇文档：")
for doc in docs:
    print(f"  - {doc.filename}  {doc.size} 字节  doc_id={doc.doc_id}")
assert docs, "data/uploads 下没有文档，请先放入 .txt/.md（见任务文档 P0-5）"

store = VectorStore(INDEX_DIR)
embedder = get_embedding_provider(settings)

for doc in docs:
    store.remove_document(doc.doc_id)  # 幂等：重跑不会重复入库
    text = load_document(doc.path)
    parts = split_text(text, settings.chunk_size, settings.chunk_overlap)
    chunks = [
        Chunk(
            chunk_id=f"{doc.doc_id}-{i:04d}",
            doc_id=doc.doc_id,
            filename=doc.filename,
            index=i,
            text=piece,
            start=start,
            end=end,
        )
        for i, (start, end, piece) in enumerate(parts)
    ]
    vectors = embedder.embed([c.text for c in chunks])
    store.add(chunks, np.asarray(vectors, dtype=np.float32))

store.save()
print("索引概况：", store.stats())

## 2. 组装 pipeline

`RAGPipeline` 只做编排：它自己不切分、不算向量、不发请求，全靠注入进来的四个组件。
所以任何一环都能替换（NFR-2）。

In [ ]:
pipeline = RAGPipeline(
    settings=settings,
    store=store,
    embedder=embedder,
    reranker=get_reranker(settings),
    llm=get_llm(settings),
)
print("rerank_enabled =", settings.rerank_enabled)

## 3. 召回：Top-20

先只看召回阶段。`pipeline.recall()` 把问题向量化，再按余弦相似度取 Top-K。

观察点：这一列是**粗筛**，追求"不漏"——所以排在后面的候选不一定真的不相关。

In [ ]:
QUESTION = "Rerank 重排的接口地址是什么？请求体里有哪些字段？"

candidates, used = pipeline.retrieve(QUESTION, doc_ids=None)

print(f"问题：{QUESTION}")
print(f"召回 {len(candidates)} 条（含 chunk_id / cosine / rank）：")
for item in candidates:
    preview = " ".join(item.chunk.text.split())[:36]
    print(f"  rank={item.rank:2d}  cosine={item.score:+.4f}  {item.chunk.chunk_id}  {preview}")

## 4. 重排：Top-4，以及排名变化

`pipeline.rerank()` 用交叉编码器（Cross-encoder）对召回的候选重新打分。

重点看 `original_rank → new_rank`：这是本项目最有价值的可观测数据——
它直接告诉你**重排把谁捞上来了**。

In [ ]:
print(f"重排后 {len(used)} 条（含 rerank 分数与排名变化）：")
for item in used:
    move = item.original_rank - item.new_rank
    arrow = "↑ 升" if move > 0 else ("↓ 降" if move < 0 else "— 不变")
    preview = " ".join(item.chunk.text.split())[:36]
    print(
        f"  new={item.new_rank}  original={item.original_rank}  {arrow:6s}"
        f"  rerank={item.rerank_score:.4f}  {item.chunk.chunk_id}  {preview}"
    )

changed = sum(1 for item in used if item.original_rank != item.new_rank)
print()
print(f"排名发生变化的条数：{changed} / {len(used)}")
assert changed > 0, "本应至少有一条内容的排名发生变化（否则重排没起作用）"

## 5. 对照实验：关掉重排会怎样

`use_rerank=False` 是给消融实验用的显式开关（§7.10）。

关掉之后，`used` 应当**就是召回结果的前 N 条**，且每条都标记 `degraded=True`
——提醒你别把它跟正常重排的分数混着比。

In [ ]:
_, used_off = pipeline.retrieve(QUESTION, doc_ids=None, use_rerank=False)

same = [u.chunk.chunk_id for u in used_off] == [
    c.chunk.chunk_id for c in candidates[: settings.rerank_top_n]
]
assert same, "use_rerank=False 时，used 应当就是召回的前 N 条"
assert all(u.degraded for u in used_off), "降级结果必须全部标记 degraded=True"
print("✓ used == candidates[:rerank_top_n]，且全部 degraded=True")

for item in used_off:
    print(
        f"  new={item.new_rank}  original={item.original_rank}"
        f"  degraded={item.degraded}  {item.chunk.chunk_id}"
    )

## 6. 验证：`doc_ids` 过滤在召回阶段生效

`doc_ids` 是在**召回**时过滤的，不是召回之后再筛——所以过滤后的候选数会明显变少。

In [ ]:
target = docs[0]
only = pipeline.recall(QUESTION, doc_ids=[target.doc_id])

print(f"只检索《{target.filename}》（doc_id={target.doc_id}）：{len(only)} 条")
print(f"不做过滤：{len(candidates)} 条")
assert only, "过滤后不该为空"
assert all(r.chunk.doc_id == target.doc_id for r in only), "过滤后出现了其它文档的 chunk"
print("✓ 命中的每一条都属于指定文档")

## 小结

- **召回**用双塔（快、粗，取 20 条），**重排**用交叉编码器（慢、准，取 4 条）。
- `original_rank → new_rank` 是判断重排有没有价值的第一手证据。
- `use_rerank` 这个显式开关，就是 §13 消融实验的基础。
- 到这里 **M2 完成**：检索链路（召回 + 重排）已经打通。下一步 T9 接上生成。